# 🔬 DataForge Arena — GRPO Training Notebook
**Autonomous Data Cleaning Agent | Theme 3.1: World Modeling**
*Runs on Google Colab T4 GPU. Expected runtime: ~45 minutes for 300 steps.*


In [ ]:
!pip install -q unsloth trl peft transformers datasets pandas numpy matplotlib
!pip install -q "git+https://github.com/huggingface/openenv.git" 2>/dev/null || pip install -q openenv


In [ ]:
import os
if not os.path.exists("dataforge-arena"):
    !git clone https://github.com/vivekyarra/dataforge-arena.git
os.chdir("dataforge-arena")
!git pull
import sys; sys.path.insert(0, ".")
print("\u2705 Repo ready")


In [ ]:
from environment.env import DataForgeEnv, DataForgeObservation
from environment.corruptor import Corruptor
from environment.reward import RewardComputer
from environment.schemas import HEALTHCARE_SCHEMA, FINANCIAL_SCHEMA, CORRUPTOR_TOOLS
import pandas as pd
clean = pd.read_csv("data/healthcare_clean.csv")
c = Corruptor(); c.force_tier(1)
env = DataForgeEnv(c, HEALTHCARE_SCHEMA, clean)
obs = env.reset()
print(f"\u2705 Environment OK | Errors: {obs.total_errors} | Violation: {getattr(obs,'violation_type','N/A')}")
print(f"   Target cell: {getattr(obs,'target_cell_hint','N/A')}")
print(f"   Column stats: {getattr(obs,'column_stats','N/A')}")


In [ ]:
from training.prompt import build_prompt
prompt = build_prompt(obs)
print("=== AGENT OBSERVATION (first 1200 chars) ===")
print(prompt[:1200])


# Training: GRPO with Constraint-Aware Reward
# The agent is trained with 6 reward signals:
# - accuracy_delta x 250: Did the repair improve ground-truth accuracy?
# - constraint_alignment: Did the agent identify the correct violation type?
# - schema_alignment: Did the agent pick the right tool for the column's data type?
# - outlier_targeting: Did the agent target a statistical outlier (z-score > 2.5)?
# - reasoning_quality: Does the agent's justification name the column and violation?
# - parse_bonus: Did the agent output clean JSON?


In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "training.train_grpo"],
    capture_output=False,
    text=True
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv("logs/training_log.csv")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("DataForge Arena \u2014 GRPO Training Results", fontsize=14, fontweight="bold")

# Plot 1: Total reward
ax = axes[0]
ax.plot(df["step"], df["total_reward"], alpha=0.25, color="#3b82f6")
ax.plot(df["step"], df["total_reward"].rolling(8, min_periods=1).mean(),
        color="#3b82f6", linewidth=2.5, label="smoothed")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5, label="zero baseline")
ax.set_xlabel("Training Step"); ax.set_ylabel("Total Reward")
ax.set_title("Reward Improvement"); ax.legend(); ax.grid(alpha=0.3)

# Plot 2: Parse success rate
ax = axes[1]
ax.plot(df["step"], df["parse_success_rate"], color="#10b981", linewidth=2.5)
ax.fill_between(df["step"], 0, df["parse_success_rate"], alpha=0.15, color="#10b981")
ax.set_xlabel("Training Step"); ax.set_ylabel("Parse Success Rate")
ax.set_title("JSON Format Learning"); ax.set_ylim(0, 1.05); ax.grid(alpha=0.3)

# Plot 3: Reward components
if "constraint_alignment" in df.columns:
    ax = axes[2]
    components = ["accuracy_delta", "constraint_alignment", "schema_alignment", "reasoning_quality"]
    colors = ["#3b82f6", "#f59e0b", "#10b981", "#8b5cf6"]
    for col, color in zip(components, colors):
        if col in df.columns:
            ax.plot(df["step"], df[col].rolling(8, min_periods=1).mean(),
                   label=col.replace("_", " "), color=color, linewidth=2)
    ax.set_xlabel("Training Step"); ax.set_ylabel("Component Reward")
    ax.set_title("Reward Components"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("logs/training_curves_final.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Plot saved to logs/training_curves_final.png")


In [ ]:
!python eval/evaluate.py --agent-mode heuristic --episodes 20 --tier 1 --steps 5 --seed 7


In [ ]:
import json
with open("eval/results.json", encoding="utf-8") as f:
    r = json.load(f)
print("=" * 55)
print("  DATAFORGE ARENA \u2014 RESULTS SUMMARY")
print("=" * 55)
print(f"  Agent mode:        {r['agent_mode']}")
print(f"  Episodes:          {r['episodes']}")
print(f"  Heuristic win rate: {r['surgeon_win_rate']:.1%}")
print(f"  Heuristic advantage over random: +{r['surgeon_advantage_accuracy_delta']:.4f}")
print(f"  Corruption types:  7 (Tier 1/2/3)")
print(f"  Reasoning layer:   Schema-grounded causal CoT")
print(f"  Reward signals:    6 (constraint/schema/distribution/reasoning/parse/anti-hack)")
print("=" * 55)
print("  See logs/training_curves_final.png for reward curves")
print("  See HF Space for live demo")
